# Training Notebook (Colab / Local)

Config-driven training pipeline. Core logic lives in `data_splitter.py`, `dataset.py`,
`model.py`, `pytorch_lightning.py`, and `training_utils.py` — this notebook just wires
them together against `configs/base.yaml`. Edit those files directly; local runs (no
`google.colab` import) pick up changes immediately, no push/pull needed.

**Kaggle auth** (only needed if `data/` isn't already present): tries, in order, an
existing `KAGGLE_API_TOKEN` env var, `~/.kaggle/access_token`, `~/.kaggle/kaggle.json`,
a Colab secret named `KAGGLE_API_TOKEN` (browser UI only), then an interactive prompt.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

# Works around a known Windows conda/pip OpenMP DLL conflict (harmless elsewhere).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

GIT_URL = 'https://github.com/hagairavid18/beilinson.git'
GIT_BRANCH = 'main'

try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content/beilinson')
    if PROJECT_ROOT.exists():
        subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull'])
    else:
        subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(PROJECT_ROOT)])
except ImportError:
    # Not on Colab (e.g. a local kernel) - use the repo checkout we're already in.
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Has data already:', any((PROJECT_ROOT / 'data').glob('*/*')))

Project root: /content/beilinson
Has data already: False


In [2]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

0

In [ ]:
import yaml
import torch
import pandas as pd
import lightning.pytorch as pl
from torch.utils.data import DataLoader
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from dataset import MultiClipWorkoutDataset, WorkoutSequenceDataset
from model import SequenceClassifier
from pytorch_lightning import WorkoutLightningModule
from training_utils import ensure_artifacts, ensure_dataset, epoch_history, evaluate_multi_clip, save_results

## Config

In [4]:
with open(PROJECT_ROOT / 'configs' / 'base.yaml', 'r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

CONFIG

{'mode': {'name': 'colab_training', 'seed': 42, 'build_artifacts': True},
 'data': {'data_dir': 'data',
  'artifacts_dir': 'artifacts',
  'sequence_len': 16,
  'image_size': 128,
  'batch_size': 16,
  'num_workers': 2,
  'train_frac': 0.7,
  'val_frac': 0.15,
  'test_frac': 0.15,
  'manifest_name': 'sequence_manifest_len16.csv'},
 'model': {'in_channels': 3,
  'hidden_dims': [32, 64, 128],
  'embedding_dim': 128,
  'dropout': 0.2,
  'temporal_pooling': 'mean'},
 'training': {'lr': 0.001,
  'weight_decay': 0.0001,
  'max_epochs': 10,
  'accelerator': 'auto',
  'devices': 'auto',
  'precision': '32-true',
  'log_every_n_steps': 10,
  'patience': 4,
  'checkpoint_dir': 'artifacts/checkpoints'}}

## Dataset

Downloads from Kaggle only if `data/` is empty (see `ensure_dataset` in `training_utils.py`
for the auth fallback chain).

In [5]:
class_names = ensure_dataset(PROJECT_ROOT)
print(f'{len(class_names)} classes:', class_names)

100%|██████████| 818M/818M [00:08<00:00, 103MB/s] 

Extracting files...


22 classes: ['barbell biceps curl', 'bench press', 'chest fly machine', 'deadlift', 'decline bench press', 'hammer curl', 'hip thrust', 'incline bench press', 'lat pulldown', 'lateral raises', 'leg extension', 'leg raises', 'plank', 'pull up', 'push up', 'romanian deadlift', 'russian twist', 'shoulder press', 'squat', 't bar row', 'tricep dips', 'tricep pushdown']


## Build dataloaders

`ensure_artifacts` builds (or reuses) the clip/frame manifests and the fixed-length
`f00..fNN` frame-sequence CSV. Each split gets its own `WorkoutSequenceDataset`, wrapped
in a `DataLoader`. Manifest frame paths are stored as `data/<class>/<file>`, relative to
`PROJECT_ROOT` — so `PROJECT_ROOT` itself (not `PROJECT_ROOT / 'data'`) is the dataset's
`data_root`.

In [6]:
pl.seed_everything(CONFIG['mode']['seed'], workers=True)

artifacts = ensure_artifacts(CONFIG, PROJECT_ROOT)

data_cfg = CONFIG['data']
image_size = data_cfg['image_size']
batch_size = data_cfg['batch_size']
num_workers = data_cfg['num_workers']
pin_memory = torch.cuda.is_available()

train_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], PROJECT_ROOT, split='train', image_size=image_size)
val_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], PROJECT_ROOT, split='val', image_size=image_size)
test_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], PROJECT_ROOT, split='test', image_size=image_size)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)

label_map = pd.read_csv(artifacts['label_map'])
num_classes = int(label_map['label_id'].nunique())

print(f'{num_classes} classes')
print(f'train {len(train_dataset)} / val {len(val_dataset)} / test {len(test_dataset)} clips')

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


22 classes
train 771 / val 166 / test 165 clips


## Build model

`SequenceClassifier` runs a small CNN (`FrameEncoder`) over every frame independently,
then pools the per-frame embeddings across time (mean/max/LSTM, per `temporal_pooling`)
before a linear classifier head. `WorkoutLightningModule` wraps it with the train/val/test/
predict steps and the optimizer.

In [7]:
model_cfg = CONFIG['model']
training_cfg = CONFIG['training']

model = SequenceClassifier(
    num_classes=num_classes,
    in_channels=model_cfg['in_channels'],
    hidden_dims=tuple(model_cfg['hidden_dims']),
    embedding_dim=model_cfg['embedding_dim'],
    dropout=model_cfg['dropout'],
    temporal_pooling=model_cfg['temporal_pooling'],
)
lit_module = WorkoutLightningModule(
    model=model,
    lr=training_cfg['lr'],
    weight_decay=training_cfg['weight_decay'],
)

lit_module

WorkoutLightningModule(
  (model): SequenceClassifier(
    (frame_encoder): FrameEncoder(
      (backbone): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (6): ReLU(inplace=True)
        (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (10): ReLU(inplace=True)
        (11): MaxPool2d(kernel_size=2, stride=2, paddi

## Build trainer

Callbacks: `ModelCheckpoint` keeps the best epoch by `monitor`/`monitor_mode`,
`EarlyStopping` stops after `patience` epochs without improvement, `LearningRateMonitor`
logs the LR each epoch. `precision: auto` picks fp16 on GPU, fp32 on CPU.

In [8]:
checkpoint_dir = PROJECT_ROOT / training_cfg.get('checkpoint_dir', 'artifacts/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

monitor = training_cfg.get('monitor', 'val_acc')
monitor_mode = training_cfg.get('monitor_mode', 'max')

callbacks = [
    ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename='epoch{epoch:02d}-{val_acc:.3f}',
        monitor=monitor,
        mode=monitor_mode,
        save_top_k=1,
    ),
    EarlyStopping(monitor=monitor, mode=monitor_mode, patience=training_cfg.get('patience', 4)),
    LearningRateMonitor(logging_interval='epoch'),
]

precision = training_cfg.get('precision', '32-true')
if precision == 'auto':
    precision = '16-mixed' if torch.cuda.is_available() else '32-true'

trainer = pl.Trainer(
    max_epochs=training_cfg.get('max_epochs', 10),
    accelerator=training_cfg.get('accelerator', 'auto'),
    devices=training_cfg.get('devices', 'auto'),
    precision=precision,
    log_every_n_steps=training_cfg.get('log_every_n_steps', 10),
    default_root_dir=str(PROJECT_ROOT / 'artifacts'),
    callbacks=callbacks,
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Train

Live progress bar with per-step loss/accuracy comes from Lightning's `Trainer.fit`
directly below.

In [ ]:
trainer.fit(lit_module, train_loader, val_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SequenceClassifier │  113 K │ train │     0 │
└───┴───────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 113 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 113 K                                                                                                
Total estimated model params size (MB): 0.452                                                                      
Modules in train mode: 25                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 3. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 6. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

## Training history

Per-epoch train/val loss and accuracy, read back from the CSV logger.

In [ ]:
epoch_history(trainer)

## Evaluate

In [ ]:
test_results = trainer.test(lit_module, dataloaders=test_loader, verbose=True)

## Predict & save

In [ ]:
prediction_batches = trainer.predict(lit_module, dataloaders=test_loader)
summary = save_results(trainer, artifacts, test_results, prediction_batches, PROJECT_ROOT)
summary

In [ ]:
import json

summary_path = PROJECT_ROOT / 'artifacts' / 'training_summary.json'
with open(summary_path, 'r', encoding='utf-8') as handle:
    summary = json.load(handle)

summary

## Multi-clip evaluation (validation set)

Checks whether averaging predictions over multiple windows per clip (instead of the single
evenly-spaced window every clip gets above) actually helps on *this* dataset, before relying
on it anywhere else. `MultiClipWorkoutDataset` splits each clip into `NUM_CLIPS` contiguous
segments and samples a window from each (see `sample_or_pad_indices_multi` in
`data_splitter.py`); `evaluate_multi_clip` averages the softmax predictions across those
windows per clip. Run on the validation set (not test) so test stays untouched for a final,
one-time check later.

In [ ]:
from dataset import MultiClipWorkoutDataset
from training_utils import epoch_history, evaluate_multi_clip

NUM_CLIPS = 5

multi_clip_val_dataset = MultiClipWorkoutDataset(
    frame_manifest_path=artifacts['frame_manifest'],
    label_map_path=artifacts['label_map'],
    data_root=PROJECT_ROOT,
    split='val',
    sequence_len=data_cfg['sequence_len'],
    num_clips=NUM_CLIPS,
    image_size=data_cfg['image_size'],
)
multi_clip_val_accuracy, multi_clip_val_results = evaluate_multi_clip(
    lit_module, multi_clip_val_dataset, batch_size=max(1, batch_size // NUM_CLIPS),
)

single_window_val_acc = float(epoch_history(trainer)['val_acc'].iloc[-1])
print(f'Single-window val_acc (last epoch): {single_window_val_acc:.4f}')
print(f'Multi-clip (num_clips={NUM_CLIPS}) val_acc:        {multi_clip_val_accuracy:.4f}')
print(f'Delta: {multi_clip_val_accuracy - single_window_val_acc:+.4f}')